In [25]:
import json

# Only works for some .json exported files from nsys-ui...
def process_json_report(filePath):
    kernel_trace = []
    kernel_names = []
    
    with open(filePath, 'r') as f:
        for line in f:
            try:
                entry = json.loads(line)
                if entry.get('type') == 79:
                    kernel_trace.append(line.get('CudaEvent'))
                    
                if entry.get('type') == 'String':
                     kernel_names.append(entry['value'])
            except json.JSONDecodeError as e:
                print(f"Skipping malformed line: {e}")
    return (kernel_trace, kernel_names)

In [28]:
import sqlite3

def process_sqlite_report(filePath):
        
    con = sqlite3.connect(filePath)
    con.row_factory = sqlite3.Row 
    cur = con.cursor()
    
    query = """
    SELECT
        names.value AS kernel_name,
        k.start,
        k.end,
        k.end - k.start AS duration,
        k.registersPerThread,
        k.gridX, k.gridY, k.gridZ,
        k.blockX, k.blockY, k.blockZ,
        k.staticSharedMemory, k.dynamicSharedMemory, k.localMemoryPerThread, k.localMemoryTotal, k.sharedMemoryLimitConfig,
        k.streamId,
        k.deviceId
    FROM CUPTI_ACTIVITY_KIND_KERNEL AS k
    JOIN StringIds AS names ON k.demangledName = names.id
    ORDER BY k.start;
    """
    cur.execute(query)
    
    rows = [dict(row) for row in cur.fetchall()]

    return rows
    

In [32]:
rows = process_sqlite_report("build/Linux-x86_64/report101_1575MHz_bindless_normal.sqlite")
print(rows[0])

{'kernel_name': 'Typeinfo name for popsift::normalizedSource::Horiz<(bool)0>', 'start': 313482745, 'end': 313878203, 'duration': 395458, 'registersPerThread': 28, 'gridX': 30, 'gridY': 2160, 'gridZ': 1, 'blockX': 128, 'blockY': 1, 'blockZ': 1, 'staticSharedMemory': 0, 'dynamicSharedMemory': 0, 'localMemoryPerThread': 0, 'localMemoryTotal': 42467328, 'sharedMemoryLimitConfig': 1, 'streamId': 39, 'deviceId': 0}


In [33]:
print(rows[0]["kernel_name"])

Typeinfo name for popsift::normalizedSource::Horiz<(bool)0>


In [1]:
import pandas as pd

df = pd.read_csv("build/Linux-x86_64/metadata_report101_1575MHz_bindless_normal.csv")
print(df)

       filename  feature_count  descriptor_count
0      0001.png          11450             14878
1      0002.png           8234             10043
2      0003.png          11494             13571
3      0004.png           4479              5405
4      0005.png          14026             18084
...         ...            ...               ...
11995  0796.png           6657              7965
11996  0797.png          14074             16629
11997  0798.png           9224             11033
11998  0799.png           1121              1302
11999  0800.png           9874             11693

[12000 rows x 3 columns]
